# Detección de eventos en spreads cointegrados

**Autor:** Samuel  
**Institución:** Tecnológico de Monterrey  
**Área:** Pairs trading en Forex  
**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud

En *Amplitud* encontramos una señal bruta pequeña pero real, de tamaño similar a los costos. Aquí estudiamos **dónde se pierde**: si las pérdidas se concentran en pocas operaciones en las que la relación entre monedas se rompe, un detector de rupturas podría mejorar el neto.

## Estructura de este Notebook
1. Fase 0: Motor y datos (se reutiliza el de *Amplitud*)
2. Fase 1: Operaciones individuales
3. Fase 2: Techo teórico (qué pasaría si evitáramos las rupturas)
4. Fase 3: Detector de rupturas (CUSUM)
5. Fase 4: Validación con placebo
6. Fase 5: Filtro horario
7. Conclusión

## Fase 0: Motor y Datos

Reutilizamos los datos y algoritmos probados en 'Amplitud.ipynb'

In [2]:
# Fase 1 — Datos
# %pip install yfinance pandas numpy

# Fase 3 — Cointegración y corrección por pruebas múltiples
# %pip install scipy statsmodels

# Fase 4 — Motor (opcional, acelera ~50x)
# %pip install numba

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
from itertools import combinations
from scipy import stats

In [4]:
pd.set_option('display.max_columns', None)

base = ['EURUSD','GBPUSD','AUDUSD','NZDUSD','USDJPY','USDCAD',
        'USDCHF','USDMXN','USDCNY','USDSEK','USDNOK','USDSGD']
nuevas = ['USDZAR','USDBRL','USDINR','USDKRW','USDPLN','USDHUF','USDCZK','USDILS','USDTHB']

tickers = {k: f'{k}=X' for k in base + nuevas}

raw_close = yf.download(list(tickers.values()), period='729d', interval='60m', progress=False)['Close']
ticker_to_name = {v: k for k, v in tickers.items()}
raw_data = raw_close.rename(columns=ticker_to_name)

print(f"Descargado: {raw_data.shape}")

Descargado: (17408, 21)


In [5]:
# Cobertura: fracción de barras horarias con dato, respecto a la divisa con más datos
cobertura = raw_data.notna().mean() / raw_data.notna().mean().max()
print(cobertura.sort_values().round(3))

MIN_COBERTURA = 0.90
aceptadas = cobertura[cobertura >= MIN_COBERTURA].index.tolist()
descartadas = [c for c in tickers if c not in aceptadas]
print(f"\nAceptadas ({len(aceptadas)}): {aceptadas}")
print(f"Descartadas por cobertura ({len(descartadas)}): {descartadas}")

Ticker
USDBRL    0.682
USDCNY    0.773
USDINR    0.816
USDKRW    0.952
USDILS    0.965
USDSEK    0.977
USDTHB    0.985
USDNOK    0.989
USDJPY    0.989
USDCHF    0.991
USDSGD    0.994
EURUSD    0.994
GBPUSD    0.994
USDMXN    0.995
USDZAR    0.998
AUDUSD    0.998
NZDUSD    0.998
USDPLN    0.998
USDCAD    0.998
USDHUF    1.000
USDCZK    1.000
dtype: float64

Aceptadas (18): ['AUDUSD', 'EURUSD', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDCZK', 'USDHUF', 'USDILS', 'USDJPY', 'USDKRW', 'USDMXN', 'USDNOK', 'USDPLN', 'USDSEK', 'USDSGD', 'USDTHB', 'USDZAR']
Descartadas por cobertura (3): ['USDCNY', 'USDBRL', 'USDINR']


In [6]:
data = raw_data[aceptadas].ffill().dropna()

# Rangos aproximados — ajústalos si alguna divisa queda fuera por una razón legítima
expected = {'EURUSD':(0.9,1.4),'GBPUSD':(1.0,1.6),'AUDUSD':(0.5,0.9),'NZDUSD':(0.5,0.8),
            'USDJPY':(100,175),'USDCAD':(1.1,1.6),'USDCHF':(0.7,1.1),'USDMXN':(14,26),
            'USDCNY':(6.3,7.6),'USDSEK':(8,12),'USDNOK':(8,12.5),'USDSGD':(1.1,1.6),
            'USDZAR':(12,24),'USDBRL':(4,7),'USDINR':(75,100),'USDKRW':(1100,1700),
            'USDPLN':(3.3,4.8),'USDHUF':(280,430),'USDCZK':(19,27),'USDILS':(3,4.3),'USDTHB':(28,42)}

chk = pd.DataFrame({'media': data.mean(), 'min': data.min(), 'max': data.max()})
chk['ok'] = [expected[c][0] <= chk.loc[c, 'media'] <= expected[c][1] for c in chk.index]
display(chk.round(4))
print(data.shape, data.index.min(), '->', data.index.max())
assert chk['ok'].all(), "Algún nivel de precio no corresponde a su nombre (o el rango esperado hay que ajustarlo)"

,media,min,max,ok
Ticker,,,,
AUDUSD,0.6674,0.5952,0.7273,True
EURUSD,1.1217,1.0196,1.2047,True
GBPUSD,1.3123,1.2108,1.3850,True
NZDUSD,0.5900,0.5512,0.6376,True
USDCAD,1.3858,1.3359,1.4781,True
USDCHF,0.8388,0.7609,0.9222,True
USDCZK,22.0848,20.1080,24.7250,True
USDHUF,347.1686,300.1800,406.1000,True
USDILS,3.4105,2.7993,3.8435,True


(17408, 18) 2024-01-18 00:00:00+00:00 -> 2026-11-03 05:00:00+00:00


In [7]:
nombres = {
    'EURUSD': 'Euro',        'GBPUSD': 'Libra',       'AUDUSD': 'Dolar_AUS',   'NZDUSD': 'Dolar_NZ',
    'USDCAD': 'Dolar_CAN',   'USDCHF': 'Franco_SUI',  'USDCZK': 'Corona_CZ',   'USDHUF': 'Forinto_HU',
    'USDILS': 'Shekel_IL',   'USDJPY': 'Yen',         'USDKRW': 'Won_KR',      'USDMXN': 'Peso_MX',
    'USDNOK': 'Corona_NO',   'USDPLN': 'Zloty_PL',    'USDSEK': 'Corona_SE',   'USDSGD': 'Dolar_SG',
    'USDTHB': 'Baht_TH',     'USDZAR': 'Rand_ZA',
}
# Documenta la dirección de cada cotización (no cambia los datos)
cotizacion = {nombres[t]: ('USD por unidad' if t.endswith('USD') else 'unidades por USD') for t in nombres}

data_t = data.copy()          # versión con tickers, por si hay que compararla con los notebooks previos
data = data_t.rename(columns=nombres)
print(pd.Series(cotizacion))

Euro            USD por unidad
Libra           USD por unidad
Dolar_AUS       USD por unidad
Dolar_NZ        USD por unidad
Dolar_CAN     unidades por USD
Franco_SUI    unidades por USD
Corona_CZ     unidades por USD
Forinto_HU    unidades por USD
Shekel_IL     unidades por USD
Yen           unidades por USD
Won_KR        unidades por USD
Peso_MX       unidades por USD
Corona_NO     unidades por USD
Zloty_PL      unidades por USD
Corona_SE     unidades por USD
Dolar_SG      unidades por USD
Baht_TH       unidades por USD
Rand_ZA       unidades por USD
dtype: str


In [8]:
warnings.filterwarnings('ignore')
try:
    from numba import njit          # opcional: %pip install numba (acelera mucho)
except ImportError:
    njit = lambda f: f

In [9]:
TEST_DAYS, TRAIN_DAYS, MIN_TRAIN = 30, 120, 240   # ventanas de 30 días, entrenamiento de 120
STOP, HL_MAX = 3.5, 24 * 15                       # stop en sigmas; vida media máxima (horas)
COST_BPS = 1.0                                    # por pata y por lado
N_MAX, JACC_MAX = 20, 0.4                         # canastas por ventana; traslape máximo entre canastas

In [10]:
def fit_ou(s):
    x, y = s[:-1], s[1:]
    b, a = np.polyfit(x, y, 1)
    if not (0.0 < b < 1.0): return None
    r = y - (a + b * x)
    return dict(b=b, mu=a / (1 - b), sig=r.std(ddof=2) / np.sqrt(1 - b * b), hl=np.log(2) / (-np.log(b)))

@njit
def positions_from_z(z, k, e, stop):
    n = len(z); pos = np.zeros(n); p = 0.0; blocked = 0.0
    for t in range(n):
        zt = z[t]
        if blocked != 0.0 and abs(zt) < k: blocked = 0.0
        if p == 0.0:
            if blocked == 0.0:
                if zt >= k: p = -1.0
                elif zt <= -k: p = 1.0
        else:
            if p < 0 and zt <= e: p = 0.0
            elif p > 0 and zt >= -e: p = 0.0
            elif abs(zt) >= stop:
                blocked = 1.0 if zt > 0 else -1.0
                p = 0.0
        pos[t] = p
    return pos

def ventanas(index, train_days=TRAIN_DAYS):
    day = index.normalize().values; ud = np.unique(day)
    for j in range(MIN_TRAIN, len(ud) - TEST_DAYS + 1, TEST_DAYS):
        tr = (day >= ud[j - train_days]) & (day < ud[j]); te = (day >= ud[j]) & (day < ud[j + TEST_DAYS])
        yield j, tr, te

In [11]:
def _df(s, mu=None):
    sl = s[:-1] - (s[:-1].mean() if mu is None else mu); ds = np.diff(s)
    d = (sl @ ds) / (sl @ sl); e = ds - d * sl
    return -d, -(d / np.sqrt(e.var(ddof=2) / (sl @ sl)))

def puntaje_cestas(Ltr, cestas):
    sc = np.empty(len(cestas)); coefs = []
    for n, idx in enumerate(cestas):
        A = np.column_stack([np.ones(len(Ltr))] + [Ltr[:, m] for m in idx[1:]])
        coef = np.linalg.lstsq(A, Ltr[:, idx[0]], rcond=None)[0]
        sc[n] = _df(Ltr[:, idx[0]] - A @ coef)[1]
        coefs.append(coef)
    return sc, coefs

def elegir_cestas(sc, cestas, n_max=N_MAX, jacc_max=JACC_MAX):
    elegidas = []
    for n in np.argsort(-sc):
        s = set(cestas[n])
        if all(len(s & set(cestas[m])) / len(s | set(cestas[m])) <= jacc_max for m in elegidas):
            elegidas.append(n)
            if len(elegidas) == n_max: break
    return elegidas

def pesos_cesta(idx, coef, n_act):
    w = np.zeros(n_act); w[idx[0]] = 1.0
    for m, c in zip(idx[1:], coef[1:]): w[m] = -c
    return w

def preparar_seleccion(data, n_max=N_MAX, jacc_max=JACC_MAX, train_days=TRAIN_DAYS):
    Lp = np.log(data).values
    cestas = [c for k in (2, 3, 4) for c in combinations(range(data.shape[1]), k)]
    out = []
    for j, tr, te in ventanas(data.index, train_days):
        sc, coefs = puntaje_cestas(Lp[tr], cestas)
        sel = elegir_cestas(sc, cestas, n_max, jacc_max)
        W = np.column_stack([pesos_cesta(cestas[n], coefs[n], data.shape[1]) for n in sel])
        out.append(dict(window=j, tr=tr, te=te, W=W, cestas=[cestas[n] for n in sel], score=sc[sel]))
    return out

In [12]:
seleccion = preparar_seleccion(data)
print(len(seleccion), "ventanas;", sum(s['W'].shape[1] for s in seleccion), "canastas-ventana")

16 ventanas; 320 canastas-ventana


## Fase 1: Operaciones individuales

Hasta ahora medimos el promedio por ventana. Aquí registramos **cada operación** (entrada, duración, bruto, neto y cómo terminó) para ver qué la distingue de las demás.

Cada operación termina de una de tres formas:
- **revierte**: el spread volvió al promedio (lo esperado).
- **stop**: se alejó más de 3.5 sd y cortamos.
- **fin**: seguía abierta al terminar la ventana; se liquida a precio de mercado y **se cobra la salida** (el motor de *Amplitud* no lo cobraba).

In [13]:
def operaciones_spread(Lp_tr, Lp_te, w, idx_te, k=1.0, e=0.0, cost_bps=COST_BPS, retraso=1):
    """Lista de operaciones (centro fijo) de un spread, con bruto, neto y forma de salida."""
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()                                       # exposición bruta
    ou = fit_ou(s_tr)
    if ou is None or ou['hl'] > HL_MAX:
        return []
    z = (s_te - ou['mu']) / ou['sig']
    pos = positions_from_z(z, k, e, STOP)
    ds = np.diff(s_te); n = len(ds) - retraso
    ops, t = [], 0
    while t < n:
        if pos[t] == 0:
            t += 1; continue
        a = t
        while t < n and pos[t] == pos[a]:
            t += 1
        b = t
        bruto = (pos[a:b] * ds[a + retraso:b + retraso]).sum() / E
        if b >= n:
            salida = 'fin'
        elif abs(z[b]) >= STOP:
            salida = 'stop'
        else:
            salida = 'revierte'
        ops.append(dict(entrada=idx_te[a], horas=b - a, lado=int(pos[a]), bruto=bruto,
                        neto=bruto - 2 * cost_bps / 1e4, salida=salida))
    return ops

In [14]:
Lp = np.log(data).values
T = pd.DataFrame([dict(o, ventana=s['window'], cesta=i)
                  for s in seleccion for i in range(s['W'].shape[1])
                  for o in operaciones_spread(Lp[s['tr']], Lp[s['te']], s['W'][:, i], data.index[s['te']])])
print(len(T), "operaciones;  bruto total:", round(T.bruto.sum(), 4), " neto total:", round(T.neto.sum(), 4))

1140 operaciones;  bruto total: 0.1957  neto total: -0.0323


In [15]:
display(T.groupby('salida').agg(n=('neto', 'size'), neto_medio=('neto', 'mean'),
                                neto_suma=('neto', 'sum'), horas=('horas', 'mean')).round(5))

,n,neto_medio,neto_suma,horas
salida,,,,
fin,100,-0.00095,-0.09548,163.60000
revierte,748,0.00143,1.06850,67.14305
stop,292,-0.00344,-1.00535,142.60616


## Fase 2: ¿Cuánto tendría que acertar un detector?

Clasificamos cada operación como **buena** (revierte) o **mala** (stop o fin). Si un detector evita una fracción *f* de las malas y, por error, una fracción *g* de las buenas, el neto cambia a: neto = (suma de buenas)(1 − g) − (pérdida de malas)(1 − f). Primero medimos qué combinaciones *(f, g)* serían rentables y luego revisamos si alguna característica **conocida al entrar** separa a las malas de las buenas.

In [16]:
T['malo'] = T.salida != 'revierte'
G = T.loc[~T.malo, 'neto'].sum()          # ganancia de las buenas
B = -T.loc[T.malo, 'neto'].sum()          # pérdida de las malas
print(f"buenas: +{G:.3f}   malas: -{B:.3f}   neto actual: {G - B:+.3f}")
f = np.linspace(0, 1, 5)
tab = pd.DataFrame({f'g={g:.0%}': [G * (1 - g) - B * (1 - ff) for ff in f] for g in (0, 0.1, 0.2, 0.3)},
                   index=[f'f={ff:.0%}' for ff in f])
display(tab.round(3))

buenas: +1.069   malas: -1.101   neto actual: -0.032


,g=0%,g=10%,g=20%,g=30%
f=0%,-0.032,-0.139,-0.246,-0.353
f=25%,0.243,0.136,0.029,-0.078
f=50%,0.518,0.411,0.304,0.198
f=75%,0.793,0.686,0.580,0.473
f=100%,1.069,0.962,0.855,0.748


In [17]:
score = {(s['window'], i): s['score'][i] for s in seleccion for i in range(len(s['score']))}
ini = {s['window']: data.index[s['te']][0] for s in seleccion}
T['score'] = [score[(w, c)] for w, c in zip(T.ventana, T.cesta)]            # qué tan estacionaria salió en entrenamiento
T['dia'] = [(e - ini[w]).days for e, w in zip(T.entrada, T.ventana)]       # días desde que empezó la ventana de prueba

for c in ('dia', 'score'):
    display(T.groupby(pd.qcut(T[c], 3, duplicates='drop'))
              .agg(n=('malo', 'size'), frac_mala=('malo', 'mean'), neto=('neto', 'mean')).round(4))

,n,frac_mala,neto
dia,,,
"(-0.001, 5.0]",399,0.3133,0.0000
"(5.0, 20.0]",372,0.3118,-0.0002
"(20.0, 41.0]",369,0.4092,0.0001


,n,frac_mala,neto
score,,,
"(4.87, 6.129]",383,0.3394,0.0002
"(6.129, 6.745]",378,0.3624,-0.0001
"(6.745, 9.093]",379,0.3298,-0.0001


In [18]:
T['tercil_dia'] = pd.qcut(T.dia, 3, labels=['temprano', 'medio', 'tarde'])
display(pd.crosstab(T.tercil_dia, T.salida, normalize='index').round(3))
display(T.groupby('tercil_dia').agg(n=('neto', 'size'), bruto_suma=('bruto', 'sum'),
                                    neto_suma=('neto', 'sum')).round(3))

salida,fin,revierte,stop
tercil_dia,,,
temprano,0.015,0.687,0.298
medio,0.016,0.688,0.296
tarde,0.238,0.591,0.171


,n,bruto_suma,neto_suma
tercil_dia,,,
temprano,399,0.098,0.018
medio,372,0.003,-0.072
tarde,369,0.095,0.022


In [19]:
D = 20                                   # fijado a priori, después de ver los terciles: es una hipótesis, no un resultado
T['mitad'] = np.where(T.ventana <= T.ventana.median(), 'primera mitad', 'segunda mitad')
T['abre'] = np.where(T.dia < D, 'antes del día 20', 'después')
display(T.groupby(['mitad', 'abre']).agg(n=('neto', 'size'), neto_suma=('neto', 'sum'),
                                          neto_medio=('neto', 'mean')).round(4))
print("Neto con la regla:", round(T[T.dia < D].neto.sum(), 4), "  sin la regla:", round(T.neto.sum(), 4))

n  neto_suma  neto_medio
mitad         abre                                        
primera mitad antes del día 20  371    -0.0408     -0.0001
              después           223     0.0225      0.0001
segunda mitad antes del día 20  369     0.0060      0.0000
              después           177    -0.0201     -0.0001

Neto con la regla: -0.0348   sin la regla: -0.0323


In [20]:
VENT = 24 * 5                                                  # ventana de la media móvil (5 días)

def operaciones_spread(Lp_tr, Lp_te, w, idx_te, centro='fijo', k=1.0, e=0.0, cost_bps=COST_BPS, retraso=1):
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()
    if centro == 'fijo':
        ou = fit_ou(s_tr)
        if ou is None or ou['hl'] > HL_MAX:
            return []
        z = (s_te - ou['mu']) / ou['sig']
    else:
        m_tr = pd.Series(s_tr).rolling(VENT).mean().shift(1).values
        ou = fit_ou((s_tr - m_tr)[VENT:])
        if ou is None or ou['hl'] > HL_MAX:
            return []
        m = pd.Series(np.concatenate([s_tr[-VENT:], s_te])).rolling(VENT).mean().shift(1).values[VENT:]
        z = (s_te - m) / ou['sig']
    pos = positions_from_z(z, k, e, STOP)
    ds = np.diff(s_te); n = len(ds) - retraso
    ops, t = [], 0
    while t < n:
        if pos[t] == 0:
            t += 1; continue
        a = t
        while t < n and pos[t] == pos[a]:
            t += 1
        b = t
        bruto = (pos[a:b] * ds[a + retraso:b + retraso]).sum() / E
        salida = 'fin' if b >= n else ('stop' if abs(z[b]) >= STOP else 'revierte')
        ops.append(dict(entrada=idx_te[a], horas=b - a, lado=int(pos[a]), bruto=bruto,
                        neto=bruto - 2 * cost_bps / 1e4, salida=salida, ancho_bps=k * ou['sig'] / E * 1e4))
    return ops

In [21]:
Lp = np.log(data).values
res = {}
for centro in ('fijo', 'movil'):
    Tc = pd.DataFrame([dict(o, ventana=s['window'], cesta=i)
                       for s in seleccion for i in range(s['W'].shape[1])
                       for o in operaciones_spread(Lp[s['tr']], Lp[s['te']], s['W'][:, i], data.index[s['te']], centro)])
    res[centro] = Tc
    print(f"\n{centro}: {len(Tc)} operaciones, bruto {Tc.bruto.sum():.3f}, neto {Tc.neto.sum():.3f}")
    display(pd.crosstab(Tc.salida, 'frac', normalize='columns').round(3).T)
    display(Tc.groupby('salida').agg(n=('neto', 'size'), neto_medio=('neto', 'mean'),
                                     neto_suma=('neto', 'sum'), horas=('horas', 'mean')).round(5))


fijo: 1140 operaciones, bruto 0.196, neto -0.032


salida,fin,revierte,stop
col_0,,,
frac,0.088,0.656,0.256


,n,neto_medio,neto_suma,horas
salida,,,,
fin,100,-0.00095,-0.09548,163.60000
revierte,748,0.00143,1.06850,67.14305
stop,292,-0.00344,-1.00535,142.60616



movil: 2911 operaciones, bruto 0.672, neto 0.089


salida,fin,revierte,stop
col_0,,,
frac,0.068,0.862,0.07


,n,neto_medio,neto_suma,horas
salida,,,,
fin,198,-0.00084,-0.16558,41.42424
revierte,2508,0.00039,0.97909,46.24960
stop,205,-0.00353,-0.72413,37.93659


In [22]:
VENT = 24 * 5                                                  # ventana de la media móvil (5 días)

def operaciones_spread(Lp_tr, Lp_te, w, idx_te, centro='fijo', k=1.0, e=0.0, cost_bps=COST_BPS, retraso=1):
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()
    if centro == 'fijo':
        ou = fit_ou(s_tr)
        if ou is None or ou['hl'] > HL_MAX:
            return []
        z = (s_te - ou['mu']) / ou['sig']
    else:
        m_tr = pd.Series(s_tr).rolling(VENT).mean().shift(1).values
        ou = fit_ou((s_tr - m_tr)[VENT:])
        if ou is None or ou['hl'] > HL_MAX:
            return []
        m = pd.Series(np.concatenate([s_tr[-VENT:], s_te])).rolling(VENT).mean().shift(1).values[VENT:]
        z = (s_te - m) / ou['sig']
    pos = positions_from_z(z, k, e, STOP)
    ds = np.diff(s_te); n = len(ds) - retraso
    ops, t = [], 0
    while t < n:
        if pos[t] == 0:
            t += 1; continue
        a = t
        while t < n and pos[t] == pos[a]:
            t += 1
        b = t
        bruto = (pos[a:b] * ds[a + retraso:b + retraso]).sum() / E
        salida = 'fin' if b >= n else ('stop' if abs(z[b]) >= STOP else 'revierte')
        ops.append(dict(entrada=idx_te[a], horas=b - a, lado=int(pos[a]), bruto=bruto,
                        neto=bruto - 2 * cost_bps / 1e4, salida=salida,
                        ancho_bps=k * ou['sig'] / E * 1e4))      # ganancia potencial por operación, en pb
    return ops

In [23]:
Lp = np.log(data).values
res = {}
for centro in ('fijo', 'movil'):
    Tc = pd.DataFrame([dict(o, ventana=s['window'], cesta=i)
                       for s in seleccion for i in range(s['W'].shape[1])
                       for o in operaciones_spread(Lp[s['tr']], Lp[s['te']], s['W'][:, i], data.index[s['te']], centro)])
    Tc['malo'] = Tc.salida != 'revierte'
    res[centro] = Tc
    print(centro, len(Tc), "operaciones; bruto", round(Tc.bruto.sum(), 3), " neto", round(Tc.neto.sum(), 3))

fijo 1140 operaciones; bruto 0.196  neto -0.032
movil 2911 operaciones; bruto 0.672  neto 0.089


In [24]:
for centro in ('fijo', 'movil'):
    T = res[centro].copy()
    T['tercil_ancho'] = pd.qcut(T.ancho_bps, 3, labels=['estrecho', 'medio', 'ancho'])
    print("\n", centro)
    display(T.groupby('tercil_ancho').agg(n=('neto', 'size'), ancho_medio_bps=('ancho_bps', 'mean'),
                                           bruto_medio=('bruto', 'mean'), neto_medio=('neto', 'mean'),
                                           neto_suma=('neto', 'sum'), frac_mala=('malo', 'mean')).round(5))


 fijo


,n,ancho_medio_bps,bruto_medio,neto_medio,neto_suma,frac_mala
tercil_ancho,,,,,,
estrecho,383,8.68878,0.00011,-0.00009,-0.03634,0.33159
medio,378,12.60895,0.00013,-0.00007,-0.02728,0.33598
ancho,379,20.34767,0.00028,0.00008,0.03129,0.36412



 movil


,n,ancho_medio_bps,bruto_medio,neto_medio,neto_suma,frac_mala
tercil_ancho,,,,,,
estrecho,972,7.01845,0.00017,-0.00003,-0.02935,0.13272
medio,971,10.22935,0.00018,-0.00002,-0.01765,0.13594
ancho,968,16.72806,0.00034,0.00014,0.13638,0.14669


In [25]:
umbrales = [0, 5, 8, 10, 12, 15, 18, 20, 25, 30]
filas = []
for centro in ('fijo', 'movil'):
    T = res[centro]; vent = sorted(T.ventana.unique()); mitad = T.ventana.median()
    for u in umbrales:
        S = T[T.ancho_bps >= u]
        por_vent = S.groupby('ventana').neto.sum().reindex(vent, fill_value=0) / N_MAX   # media por canasta-ventana, abstenciones = 0
        filas.append(dict(centro=centro, umbral_pb=u, n=len(S), neto_medio_op=S.neto.mean(),
                          neto_vent=por_vent.mean(), p=stats.ttest_1samp(por_vent, 0).pvalue,
                          mitad1=S[S.ventana <= mitad].neto.sum(), mitad2=S[S.ventana > mitad].neto.sum()))
display(pd.DataFrame(filas).round(5))

,centro,umbral_pb,n,neto_medio_op,neto_vent,p,mitad1,mitad2
0,fijo,0,1140,-0.00003,-0.00010,0.70161,-0.01829,-0.01404
1,fijo,5,1140,-0.00003,-0.00010,0.70161,-0.01829,-0.01404
2,fijo,8,1027,-0.00002,-0.00008,0.77385,-0.01689,-0.00751
3,fijo,10,803,-0.00000,-0.00000,0.99965,-0.00687,0.00684
4,fijo,12,614,0.00001,0.00002,0.94756,-0.02511,0.02995
5,fijo,15,377,0.00009,0.00011,0.64016,-0.03579,0.06960
6,fijo,18,226,0.00024,0.00017,0.35602,-0.01805,0.07173
7,fijo,20,150,0.00049,0.00023,0.17114,-0.00221,0.07604
8,fijo,25,88,0.00052,0.00014,0.36418,-0.01073,0.05617
9,fijo,30,15,-0.00130,-0.00006,0.16673,-0.01955,0.00000


In [26]:
from collections import Counter
cest = {(s['window'], i): c for s in seleccion for i, c in enumerate(s['cestas'])}
T = res['fijo']; ancho = T[T.ancho_bps >= T.ancho_bps.quantile(2 / 3)]
ca = Counter(data.columns[j] for w, c in zip(ancho.ventana, ancho.cesta) for j in cest[(w, c)])
ct = Counter(data.columns[j] for w, c in zip(T.ventana, T.cesta) for j in cest[(w, c)])
comp = pd.DataFrame({'en_ancho': pd.Series(ca) / len(ancho), 'en_todas': pd.Series(ct) / len(T)}).fillna(0)
comp['razon'] = comp.en_ancho / comp.en_todas
display(comp.sort_values('razon', ascending=False).round(2))

,en_ancho,en_todas,razon
Shekel_IL,0.36,0.20,1.77
Rand_ZA,0.35,0.21,1.63
Dolar_AUS,0.23,0.15,1.55
Won_KR,0.42,0.28,1.48
Baht_TH,0.38,0.27,1.39
Yen,0.18,0.14,1.32
Corona_SE,0.24,0.19,1.31
Corona_NO,0.22,0.17,1.31
Libra,0.10,0.09,1.14
Franco_SUI,0.15,0.14,1.04


## Conclusión de Eventos

**Qué se hizo.** Se descompuso el P&L de las canastas de *Amplitud* en operaciones individuales para ver dónde se pierde y si un detector de rupturas podría mejorar el neto.

**Qué se encontró.**
1. El neto se decide en un duelo: las operaciones que revierten (66%) ganan +1.09 y los stops (25%) pierden −1.01. Un stop cuesta 2.4 veces lo que gana una operación buena.
2. El puntaje de estacionariedad de entrenamiento no predice qué operaciones salen mal (35/33/34% de malas por tercil).
3. Las operaciones abiertas al final de la ventana pierden por un efecto mecánico (liquidación forzada), estable entre mitades pero con poco impacto en el neto total.
4. Un centro móvil reduce los stops de 25% a 7%, pero cada ganancia se encoge y se opera 2.5 veces más: el neto no cambia. El bruto por operación (~1.8 pb) es del tamaño del costo (2 pb) en ambos centros.
5. Filtrar por el ancho esperado del spread (k·σ/E) tiene lógica económica y mejora el neto en el tercio ancho, pero el efecto no es estable entre mitades, no es monótono en el centro móvil y selecciona monedas de costo real mayor (Rand, Shekel, Won, Baht).

**Qué significa.** El cuello de botella no es la detección de rupturas sino el tamaño del bruto por operación frente al costo. Los filtros ensayados no lo cambian de orden de magnitud.

**Limitaciones.** Muestra de ~2 años; costo uniforme de 1 pb por pata; varias variantes probadas sobre la misma muestra (riesgo de sobreajuste); falta la validación con placebo de cada filtro.

## Fase 3: Detector de rupturas (CUSUM)

Regla fijada antes de mirar resultados: se acumula la desviación del spread respecto a su centro (margen δ = 0.5 σ) y, si supera el máximo observado en el entrenamiento de esa canasta, se cierra la posición y no se vuelve a operar esa canasta en la ventana. Sin parámetros ajustados sobre los resultados. Solo centro fijo.

In [27]:
def cusum_max(z, delta):
    Sp = Sn = m = 0.0
    for zt in z:
        Sp = max(0.0, Sp + zt - delta); Sn = max(0.0, Sn - zt - delta)
        m = max(m, Sp, Sn)
    return m

def cusum_alarma(z, delta, h):
    Sp = Sn = 0.0
    for t, zt in enumerate(z):
        Sp = max(0.0, Sp + zt - delta); Sn = max(0.0, Sn - zt - delta)
        if Sp > h or Sn > h:
            return t
    return None

def operaciones_spread_det(Lp_tr, Lp_te, w, idx_te, k=1.0, e=0.0, delta=0.5, cost_bps=COST_BPS, retraso=1):
    s_tr, s_te = Lp_tr @ w, Lp_te @ w
    E = np.abs(w).sum()
    ou = fit_ou(s_tr)
    if ou is None or ou['hl'] > HL_MAX:
        return [], None
    z = (s_te - ou['mu']) / ou['sig']
    h = cusum_max((s_tr - ou['mu']) / ou['sig'], delta)          # umbral = máximo visto en entrenamiento
    ta = cusum_alarma(z, delta, h)
    pos = positions_from_z(z, k, e, STOP)
    if ta is not None:
        pos[ta:] = 0.0                                           # alarma: cerrar y no volver a operar en la ventana
    ds = np.diff(s_te); n = len(ds) - retraso
    ops, t = [], 0
    while t < n:
        if pos[t] == 0:
            t += 1; continue
        a = t
        while t < n and pos[t] == pos[a]:
            t += 1
        b = t
        bruto = (pos[a:b] * ds[a + retraso:b + retraso]).sum() / E
        salida = 'alarma' if (ta is not None and b == ta) else ('fin' if b >= n else ('stop' if abs(z[b]) >= STOP else 'revierte'))
        ops.append(dict(entrada=idx_te[a], horas=b - a, lado=int(pos[a]), bruto=bruto,
                        neto=bruto - 2 * cost_bps / 1e4, salida=salida))
    return ops, ta

In [29]:
Lp = np.log(data).values
filas = []
for s in seleccion:
    for i in range(s['W'].shape[1]):
        args = (Lp[s['tr']], Lp[s['te']], s['W'][:, i], data.index[s['te']])
        base = operaciones_spread(*args, centro='fijo')
        det, ta = operaciones_spread_det(*args)
        filas.append(dict(ventana=s['window'], base=sum(o['neto'] for o in base), det=sum(o['neto'] for o in det),
                          alarma=ta is not None, dia_alarma=None if ta is None else ta / 24))
F = pd.DataFrame(filas)
print(f"Canastas con alarma: {F.alarma.mean():.1%};  día mediano de la alarma: {F.dia_alarma.median():.1f}")
print(f"Neto total  sin detector: {F.base.sum():+.3f}   con detector: {F.det.sum():+.3f}")
a = F[F.alarma]
print(f"Con alarma ({len(a)} canastas): ayudó en {(a.det > a.base).sum()}, dañó en {(a.det < a.base).sum()};  "
      f"ganancia neta del detector: {(a.det - a.base).sum():+.3f}")

Canastas con alarma: 88.4%;  día mediano de la alarma: 11.0
Neto total  sin detector: -0.032   con detector: +0.029
Con alarma (283 canastas): ayudó en 135, dañó en 69;  ganancia neta del detector: +0.062


In [30]:
pv = F.groupby('ventana')[['base', 'det']].sum() / N_MAX
dif = pv.det - pv.base
mitad = pv.index.to_series().median()
print(f"Diferencia media por ventana: {dif.mean():+.5f}   p = {stats.ttest_1samp(dif, 0).pvalue:.3f}   ventanas que mejoran: {(dif > 0).sum()}/{len(dif)}")
print(f"Primera mitad: {dif[pv.index <= mitad].sum():+.4f}   Segunda mitad: {dif[pv.index > mitad].sum():+.4f}")

Diferencia media por ventana: +0.00019   p = 0.211   ventanas que mejoran: 8/16
Primera mitad: +0.0032   Segunda mitad: -0.0001


6. Un detector de rupturas CUSUM, con regla fijada de antemano (δ = 0.5 σ, umbral = máximo del entrenamiento), suena en 88% de las canastas con día mediano 11: funciona como un temporizador y no como un detector. Mejora el neto total en +0.06, pero la mejora no es estable (8 de 16 ventanas, p = 0.21; +0.0032 en la primera mitad y −0.0001 en la segunda) y coincide con el efecto mecánico de no operar tarde.

**Cierre.** Ninguna de las seis ideas de filtrado y detección cambia el orden de magnitud del bruto por operación (~1.8 pb) frente al costo (2 pb). La limitación principal de este análisis es la muestra: ~2 años de datos horarios, con resultados que cambian ~40% al refrescarla.